# Aula 2 — Ambiente Conda e BLAST

Disciplina: **EQM — Bioinformática e Biologia Molecular**

Fluxo da prática:

**necessidade → ambiente → instalação → verificação → execução → interpretação**

Nesta aula vamos preparar o ambiente computacional e instalar **NCBI BLAST+** somente quando ele se tornar necessário.

## 1. Por que usar ambientes?

Um ambiente Conda isola programas e dependências. Isso reduz conflitos e facilita a reprodutibilidade.

Estratégia da disciplina:

- `bioinfo` — ambiente geral para ferramentas instaladas conforme a necessidade;
- `phyluce` — ambiente específico quando chegarmos às análises de UCEs.

Em um terminal comum usaríamos `conda activate bioinfo`.

No Colab, como comandos `!` rodam em shells separados, a ativação não permanece entre células. Por isso usaremos principalmente `conda run -n bioinfo comando`.

## 2. Instalar Conda no Google Colab

Execute esta célula primeiro. O runtime pode reiniciar automaticamente.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Verificar Conda

In [ ]:
!conda --version
!which conda

## 4. Configurar canais

Usaremos **conda-forge** e **Bioconda** ao longo da disciplina.

In [ ]:
!conda config --add channels conda-forge
!conda config --add channels bioconda
!conda config --set channel_priority strict
!conda config --show channels

## 5. Criar o ambiente `bioinfo`

In [ ]:
!conda create -y -n bioinfo python=3.11
!conda env list

## 6. Demonstrar ativação

Esta célula mostra como `conda activate` funciona dentro de um mesmo shell.

In [ ]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate bioinfo
echo "Ambiente ativo: $CONDA_DEFAULT_ENV"
python --version

## 7. Surge uma necessidade: comparar uma sequência

Temos uma sequência FASTA e queremos procurar sequências semelhantes no banco público. Agora faz sentido instalar BLAST.

In [ ]:
!conda install -y -n bioinfo blast
!conda run -n bioinfo blastn -version

## 8. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
BASE = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular/02_blast")
BASE.mkdir(parents=True, exist_ok=True)
print(BASE)

## 9. Recuperar a sequência `MK270576.1`

In [ ]:
import urllib.request

accession = "MK270576.1"
url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    f"?db=nuccore&id={accession}&rettype=fasta&retmode=text"
)
query = BASE / f"{accession}.fasta"
urllib.request.urlretrieve(url, query)
print(query.read_text()[:500])

## 10. Executar BLASTn remoto

A saída tabular terá: query, accession do hit, identidade, tamanho do alinhamento, cobertura, E-value, bit score e título.

In [ ]:
saida = BASE / "blastn_nt.tsv"

cmd = (
    f"blastn -query '{query}' -db nt -remote "
    f"-max_target_seqs 15 "
    f'-outfmt "6 qseqid sacc pident length qcovs evalue bitscore stitle" '
    f"-out '{saida}'"
)

!conda run -n bioinfo bash -c "$cmd"
print(saida)

## 11. Ler a saída

In [ ]:
import pandas as pd

cols = [
    "query","accession_hit","identity_pct","alignment_length",
    "query_coverage_pct","evalue","bitscore","title"
]
df = pd.read_csv(saida, sep="\t", names=cols)
df.head(15)

## 12. Interpretar

Observe:
- identidade;
- cobertura da query;
- E-value;
- bit score;
- anotação do registro.

BLAST procura similaridade. O melhor hit não deve ser tratado automaticamente como identificação definitiva.

In [ ]:
df[
    ["accession_hit","identity_pct","query_coverage_pct","evalue","bitscore","title"]
].head(10)

## 13. Registrar o ambiente

In [ ]:
env_file = BASE / "bioinfo_environment.yml"
!conda env export -n bioinfo > "$env_file"
print(env_file)

# Resultado

Ao final da aula, o estudante deverá saber:
- o que é Conda;
- o que é um ambiente;
- por que usamos Bioconda;
- como instalar uma ferramenta conforme a necessidade;
- como verificar a instalação;
- como executar BLAST dentro do ambiente correto;
- como interpretar identidade, cobertura, E-value e bit score.

**Próxima aula:** SRA → FASTQ → reads paired-end.